In [0]:
%pip install openpyxl

In [0]:
import logging
import os
import pandas as pd

In [0]:
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("parish_census_orchestrator")

In [0]:
try:

    logger.info("Starting Bronze ingestion")
    dbutils.notebook.run("/Workspace/Users/c.a.dupee@gmail.com/parish_census_portfolio/01_bronze_ingestion", 0)

    logger.info("Starting Silver transformation and QA")
    dbutils.notebook.run("/Workspace/Users/c.a.dupee@gmail.com/parish_census_portfolio/02_silver_transforms", 0)

    logger.info("Starting Gold mart generation")
    dbutils.notebook.run("/Workspace/Users/c.a.dupee@gmail.com/parish_census_portfolio/03_gold_marts", 0)

    logger.info("Generating census report view")
    dbutils.notebook.run("/Workspace/Users/c.a.dupee@gmail.com/parish_census_portfolio/04_report", 0)

    logger.info("Writing report to .xlsx")
    report_df = spark.table("workspace.default.diocesan_census_report")
    pandas_report =report_df.toPandas()
    local_workspace_path = os.path.join(os.getcwd(), "diocesan_census_report.xlsx")
    pandas_report.to_excel(local_workspace_path, index=False, engine="openpyxl")

# 2. Add 'file:' prefix so dbutils understands it is a local workspace file
    local_dbutils_source = f"file:{local_workspace_path}"

# 3. Copy it to your permanent cloud storage (DBFS or Unity Catalog Volume)
    target_storage_path = "/Volumes/workspace/default/parish_data/diocesan_census_report.xlsx" 
# OR if using Unity Catalog Volumes: "/Volumes/my_catalog/my_schema/my_volume/diocesan_census_report.xlsx"

    dbutils.fs.cp(local_dbutils_source, target_storage_path)



    logger.info("Parish census report written to .xlsx")



except Exception:

    logger.exception("Parish census pipeline failed")
    raise

else:

    logger.info("Parish census pipeline completed successfully")

